In [ ]:
## LLM CALL

In [2]:
import os

from langchain_openai import ChatOpenAI


def strip_think_tags(text: str) -> str:
    return text.split("</think>", 1)[1].strip() if "</think>" in text else text


model = os.getenv("LM_STUDIO_MODEL", "qwen/qwen3-1.7b-mlx")
base_url = os.getenv("LM_STUDIO_BASE_URL", "http://127.0.0.1:1234/v1")
api_key = os.getenv("LM_STUDIO_API_KEY", "abc123")

llm = ChatOpenAI(
    model=model,
    base_url=base_url,
    api_key=api_key,
    temperature=0.1,
)

prompt = "Explain Kubernetes in maximum 4 sentences.."

response = llm.invoke(prompt)
content = strip_think_tags(response.content)

print("Prompt:", prompt)
print("Odpowiedz:", content)

if response.usage_metadata:
    print("Input tokens:", response.usage_metadata.get("input_tokens"))
    print("Output tokens:", response.usage_metadata.get("output_tokens"))


Prompt: Explain Kubernetes in maximum 4 sentences..
Odpowiedz: 

Kubernetes is an open-source platform for automating the deployment, scaling, and management of containerized applications. It ensures reliable and scalable operations by orchestrating containers across multiple nodes, handling tasks like resource allocation and fault tolerance. Key components include pods, services, and replication controllers, which work together to maintain application consistency. It is widely used in cloud-native environments to streamline infrastructure and improve application resilience.
Input tokens: 17
Output tokens: 260


In [ ]:
## STRUCTURED OUTPUT

In [3]:
from openai import OpenAI
from pydantic import BaseModel

client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

class Ticket(BaseModel):
    name: str
    priority: str
    reason: str

response = client.chat.completions.create(
    model="openai/gpt-oss-20b", 
    messages=[
        {
            "role": "system",
            "content": (
                "Extract ticket details from the user message. "
                "Details: name, priority, reason."
            ),
        },
        {
            "role": "user",
            "content": "John reported a production outage. This is urgent.",
        },
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "ticket",
            "schema": Ticket.model_json_schema(),
        },
    },
    temperature=0,
)

raw_json = response.choices[0].message.content

ticket = Ticket.model_validate_json(raw_json)
print(f"Raw: {raw_json}")
print(f"Non raw: name: {ticket.name}, priority: {ticket.priority}, reason: {ticket.reason}")


# The output:
# Raw: {"name":"John","priority":"urgent","reason":"production outage"}
# Non raw: name: John, priority: urgent, reason: production outage

Raw: {"name":"John","priority":"Urgent","reason":"Production outage"}
Non raw: name: John, priority: Urgent, reason: Production outage


In [ ]:
## LANGGRAPH

In [4]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict

class State(TypedDict): # Shared state moving through the workflow
    text: str

def add_context(state: State): # First node
    # The entire AI magic happens here or in another node...
    return {"text": state["text"] + " -> context added"}

def summarize(state: State): # Second node
    # Another AI magic here...
    return {"text": state["text"] + " -> summarized"}

graph = StateGraph(State) # Create graph

# Add nodes
graph.add_node("add_context", add_context)
graph.add_node("summarize", summarize)

# Define workflow edges
graph.add_edge(START, "add_context")
graph.add_edge("add_context", "summarize")
graph.add_edge("summarize", END)

app = graph.compile() # Compile graph into executable workflow
result = app.invoke({"text": "user question"})
print(result)

# Output
# {"text": "user question -> context added -> summarized"}




{'text': 'user question -> context added -> summarized'}


In [ ]:
## TOOL CALL

In [8]:
import json
import os

from openai import OpenAI


ORDERS = {
    "123": {
        "status": "shipped",
        "delivery_date": "2026-05-19",
    },
    "456": {
        "status": "processing",
        "delivery_date": "2026-05-23",
    },
}


def get_order_status(order_id: str) -> str:
    order = ORDERS.get(order_id)
    if not order:
        return f"Order {order_id} was not found."
    return order["status"]


def get_delivery_date(order_id: str) -> str:
    order = ORDERS.get(order_id)
    if not order:
        return f"Order {order_id} was not found."
    return order["delivery_date"]


TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_order_status",
            "description": "Get the current status of an order.",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {
                        "type": "string",
                        "description": "The order id, for example 123.",
                    }
                },
                "required": ["order_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_delivery_date",
            "description": "Get the expected delivery date of an order.",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {
                        "type": "string",
                        "description": "The order id, for example 123.",
                    }
                },
                "required": ["order_id"],
            },
        },
    },
]


AVAILABLE_FUNCTIONS = {
    "get_order_status": get_order_status,
    "get_delivery_date": get_delivery_date,
}


def main() -> None:
    client = OpenAI(
        base_url=os.getenv("LM_STUDIO_BASE_URL", "http://127.0.0.1:1234/v1"),
        api_key=os.getenv("LM_STUDIO_API_KEY", "lm-studio"),
    )
    model = os.getenv("LM_STUDIO_MODEL", "openai/gpt-oss-20b")

    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant. Use tools when they are needed.",
        },
        {
            "role": "user",
            "content": "Has order 123 already shipped, and when will it arrive?",
        },
    ]

    while True:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
            temperature=0,
        )

        assistant_message = response.choices[0].message
        messages.append(assistant_message)

        if not assistant_message.tool_calls:
            print(assistant_message.content)
            break

        for tool_call in assistant_message.tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)
            function_result = AVAILABLE_FUNCTIONS[function_name](**function_args)

            print(f"Calling tool: {function_name}({function_args})")
            print(f"Tool result: {function_result}")

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": function_result,
                }
            )

if __name__ == "__main__":
    main()

Calling tool: get_order_status({'order_id': '123'})
Tool result: shipped
Calling tool: get_delivery_date({'order_id': '123'})
Tool result: 2026-05-19
Order **123** has already shipped.  
It is expected to arrive on **May 19, 2026**.


In [ ]:
## HUGGING FACE

In [ ]:
from huggingface_hub import InferenceClient
from textwrap import fill

token = 'hf...'


client = InferenceClient(
    provider="auto",                 # HF sam dobierze dostawcę
    api_key=token  # token z Hugging Face
)

response = client.chat.completions.create(
    model="openai/gpt-oss-120b:novita",
    messages=[
        {"role": "system", "content": "Respond briefly and to the point."},
        {"role": "user", "content": "Write 3 sentences about Azure Databricks."}
    ],
    max_tokens=1100,
)

print(fill(f"Content: {response.choices[0].message.content}", width=100))
print(f"Response usage: {response.usage}")
print(f"Finish reason: {response.choices[0].finish_reason}")

Content: Azure Databricks is an Apache Spark‑based analytics platform fully integrated with
Microsoft Azure, providing collaborative notebooks, auto‑scaling clusters, and built‑in security. It
enables data engineers and data scientists to ingest, transform, and model large datasets using
languages like Python, Scala, SQL, and R. With native connectivity to Azure services (e.g., ADLS,
Synapse, Power BI), it streamlines end‑to‑end data pipelines and machine‑learning workflows.
Response usage: ChatCompletionOutputUsage(completion_tokens=128, prompt_tokens=90, total_tokens=218, prompt_tokens_details=None, completion_tokens_details=None)
Finish reason: stop


In [ ]:
from textwrap import fill
import time
from huggingface_hub import InferenceClient

token = 'hf...'

client = InferenceClient(api_key=token, provider="auto")

models = [
    "openai/gpt-oss-20b",
    "openai/gpt-oss-120b:novita",
]

prompt = "Write 3 sentences about Azure Databricks. Only specifics, no marketing."

for model in models:
    start = time.time()
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "Respond briefly and to the point."},
                {"role": "user", "content": prompt},
            ],
            max_tokens=10000 if "120b" in model else 300,
        )
        elapsed = time.time() - start
        choice = response.choices[0]

        print("=" * 80)
        print("MODEL:", model)
        print("TIME:", round(elapsed, 2), "s")
        print("FINISH:", choice.finish_reason)
        print(fill(f"USAGE: {response.usage}", width=100))
        print(fill(f"CONTENT: {repr(choice.message.content)}", width=100))

    except Exception as e:
        print("=" * 80)
        print("MODEL:", model)
        print("ERROR:", e)

MODEL: openai/gpt-oss-20b
TIME: 2.4 s
FINISH: stop
USAGE: ChatCompletionOutputUsage(completion_tokens=245, prompt_tokens=97, total_tokens=342,
queue_time=1.6725104750000002, prompt_time=0.004760184, completion_time=0.37430762,
total_time=0.379067804, completion_tokens_details={'reasoning_tokens': 133})
CONTENT: '1. Azure Databricks is an Apache Spark‑based analytics platform integrated with Azure that
delivers a collaborative workspace for data engineers, data scientists, and analysts.  \n2. It
supports Python, Scala, R, and SQL, and includes automated cluster provisioning, dynamic scaling,
and job scheduling.  \n3. Data can be ingested from Azure Blob Storage, ADLS, Cosmos DB, or Event
Hubs, and outputs are typically written to Azure Data Lake, Synapse Analytics, or Azure SQL Data
Warehouse.'
MODEL: openai/gpt-oss-120b:novita
TIME: 5.08 s
FINISH: stop
USAGE: ChatCompletionOutputUsage(completion_tokens=153, prompt_tokens=96, total_tokens=249,
prompt_tokens_details=None, completion_toke